# 06 -- Design Catalog

Explore the trade-off space between aerodynamic performance and manufacturability:

1. **Build catalog** — baseline, v2 optimized, and alpha-blended designs
2. **Aero evaluation** — 10-target surrogate (with CG correction + control derivatives)
3. **Manufacturability scoring** — curvature, thickness, mold complexity
4. **Pareto analysis** — L/D vs. manufacturability trade-off front
5. **Geometric comparison** — planform overlays, radar charts
6. **STEP export** — batch export for downstream structural/manufacturing pipeline

In [1]:
import sys
sys.path.insert(0, '..')

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from src.config import load_all
from src.optimization.catalog import DesignCatalog
from src.evaluation.manufacturability import compute_manufacturability
from src.visualization.comparison import (
    plot_pareto, plot_radar, plot_planform_overlay, plot_summary_table,
)

from src.visualization.style import apply_style, COLORS
apply_style()

cfg = load_all()
mission = cfg['mission']
feasibility = cfg['feasibility']

## 1. Build the Design Catalog

Two reference points + 4 interpolated designs spanning the full trade-off spectrum.

In [2]:
catalog = DesignCatalog()

# ── Reference designs ──
catalog.add_baseline()

# Load v2 optimized design
try:
    catalog.add_optimized('../output/best_x_v2.npy')
    print('Using v2 optimized design')
except FileNotFoundError:
    catalog.add_optimized('../output/best_x.npy')
    print('Using v1 design (fallback)')

# ── Top feasible designs from the LHS dataset ──
from src.optimization.database import EvaluationDatabase

db = EvaluationDatabase.load('../data/eval_database.json')
X_arr, results = db.to_arrays()

# Find top 5 feasible designs by L/D (diverse)
feasible_idx = [i for i, r in enumerate(results) 
                if r.get('is_feasible', False) and r.get('L_over_D', 0) > 0]
feasible_idx.sort(key=lambda i: -results[i]['L_over_D'])

# Add top 5 with diversity filter (min distance between designs)
from src.parameterization.design_variables import get_bounds_arrays
lb, ub = get_bounds_arrays()
diag = np.linalg.norm(ub - lb)
selected = []
for idx in feasible_idx:
    x = X_arr[idx]
    # Skip if too close to an already selected design
    if any(np.linalg.norm(x - s) / diag < 0.05 for s in selected):
        continue
    selected.append(x)
    r = results[idx]
    name = f'lhs_top{len(selected)}'
    catalog.add(name, x, origin='dataset', tags=['lhs', 'feasible'])
    if len(selected) >= 5:
        break

print(f'Added {len(selected)} top feasible designs from LHS dataset')

# ── Alpha-blends between best LHS and optimized ──
if len(selected) > 0:
    best_lhs = f'lhs_top1'
    catalog.interpolate(best_lhs, 'optimized', [0.33, 0.67])

print(f'\nCatalog: {len(catalog)} designs')
for e in catalog:
    p = e.params
    print(f'  {e.name:20s}  span={2*p.half_span:.2f}m  t/c={p.body_tc_root:.3f}  '
          f'taper={p.taper_ratio:.3f}  sweep={p.le_sweep_deg:.1f}')

Using v2 optimized design
Added 5 top feasible designs from LHS dataset

Catalog: 9 designs
  baseline              span=1.50m  t/c=0.180  taper=0.150  sweep=26.0
  optimized             span=1.38m  t/c=0.247  taper=0.151  sweep=35.0
  lhs_top1              span=1.61m  t/c=0.149  taper=0.225  sweep=33.7
  lhs_top2              span=1.72m  t/c=0.205  taper=0.142  sweep=28.8
  lhs_top3              span=1.69m  t/c=0.157  taper=0.211  sweep=26.9
  lhs_top4              span=1.97m  t/c=0.151  taper=0.360  sweep=25.6
  lhs_top5              span=1.16m  t/c=0.240  taper=0.265  sweep=30.7
  blend_33              span=1.53m  t/c=0.181  taper=0.201  sweep=34.1
  blend_67              span=1.46m  t/c=0.214  taper=0.176  sweep=34.6


## 2. Aerodynamic Evaluation (Surrogate)

Instant evaluation via the pre-trained MLP ensemble (7 VLM primitives → full reconstruction).

In [3]:
catalog.evaluate_aero(
    mission=mission,
    feasibility=feasibility,
    use_surrogate=True,
    surrogate_path='../models/surrogate_v2_ctrl',
)

  baseline              L/D=-12.87  X
  optimized             L/D= 13.44  OK
  lhs_top1              L/D= 13.66  X
  lhs_top2              L/D= 13.51  X
  lhs_top3              L/D= 13.35  X
  lhs_top4              L/D= 10.21  X
  lhs_top5              L/D= 10.09  X
  blend_33              L/D= 12.70  X
  blend_67              L/D= 12.45  X


## 3. Manufacturability Scoring

Geometric metrics quantifying fabrication difficulty: twist/dihedral gradients, minimum thickness, mold complexity, taper severity.

In [4]:
catalog.evaluate_manufacturability()

# Detailed breakdown for baseline vs optimized
for name in ['baseline', 'optimized']:
    e = catalog[name]
    mm = e.manufacturing_metrics
    print(f'\n=== {name.upper()} (score={mm["manufacturability_score"]:.3f}) ===')
    for k, v in mm.items():
        if k.startswith('sub_'):
            print(f'  {k[4:]:25s}  {v:.3f}')
    print(f'  {"twist_gradient_max":25s}  {mm["twist_gradient_max"]:.1f} °/m')
    print(f'  {"dihedral_gradient_max":25s}  {mm["dihedral_gradient_max"]:.1f} °/m')
    print(f'  {"thickness_tip_mm":25s}  {mm["thickness_tip_mm"]:.1f} mm')
    print(f'  {"n_dihedral_breaks":25s}  {mm["n_dihedral_breaks"]}')

  baseline              manuf=0.631
  optimized             manuf=0.259
  lhs_top1              manuf=0.373
  lhs_top2              manuf=0.314
  lhs_top3              manuf=0.363
  lhs_top4              manuf=0.367
  lhs_top5              manuf=0.518
  blend_33              manuf=0.360
  blend_67              manuf=0.322

=== BASELINE (score=0.631) ===
  twist_smoothness           0.593
  dihedral_smoothness        0.159
  dihedral_simplicity        0.667
  tip_robustness             0.912
  taper_simplicity           0.412
  blend_smoothness           0.857
  sweep_continuity           0.778
  twist_simplicity           0.900
  equipment_space            0.169
  mold_size                  1.000
  twist_gradient_max         11.1 °/m
  dihedral_gradient_max      88.9 °/m
  thickness_tip_mm           7.6 mm
  n_dihedral_breaks          1

=== OPTIMIZED (score=0.259) ===
  twist_smoothness           0.000
  dihedral_smoothness        0.000
  dihedral_simplicity        0.333
  tip_robustn

## 4. Summary Table

In [5]:
from src.geometry.control_surfaces import compute_control_surface_geometry

print(catalog.summary())
print()

# Pandas DataFrame for detailed comparison
df = catalog.to_dataframe()

# Add control surface dimension columns
elevon_spans = []
aileron_spans = []
for e in catalog:
    cs_list = compute_control_surface_geometry(e.params)
    ev_span = 0.0
    ai_span = 0.0
    for cs in cs_list:
        if 'elevon' in cs.name.lower():
            ev_span = cs.span * 100  # m -> cm
        elif 'aileron' in cs.name.lower():
            ai_span = cs.span * 100  # m -> cm
    elevon_spans.append(ev_span)
    aileron_spans.append(ai_span)

df['elevon_span_cm'] = elevon_spans
df['aileron_span_cm'] = aileron_spans

cols = ['origin', 'L_over_D', 'static_margin', 'is_feasible',
        'manuf_manufacturability_score', 'struct_mass', 'endurance_min',
        'half_span', 'body_tc_root', 'taper_ratio',
        'elevon_deflection', 'Cl_beta', 'x_cg_frac',
        'elevon_span_cm', 'aileron_span_cm']
display(df[[c for c in cols if c in df.columns]].round(3))

DesignCatalog: 9 designs

Name                 Origin            L/D  Feas  Manuf Tags
------------------------------------------------------------
baseline             default        -12.87     X  0.631 baseline, reference
optimized            optimization    13.44    OK  0.259 optimized, reference, feasible
lhs_top1             dataset         13.66     X  0.373 lhs, feasible
lhs_top2             dataset         13.51     X  0.314 lhs, feasible
lhs_top3             dataset         13.35     X  0.363 lhs, feasible
lhs_top4             dataset         10.21     X  0.367 lhs, feasible
lhs_top5             dataset         10.09     X  0.518 lhs, feasible
blend_33             interpolation   12.70     X  0.360 blend, alpha=0.33
blend_67             interpolation   12.45     X  0.322 blend, alpha=0.67



,origin,L_over_D,static_margin,is_feasible,manuf_manufacturability_score,struct_mass,endurance_min,half_span,body_tc_root,taper_ratio,elevon_deflection,Cl_beta,elevon_span_cm,aileron_span_cm
name,,,,,,,,,,,,,,
baseline,default,-12.869,-0.294,False,0.631,0.715,6.013,0.750,0.180,0.150,-25.0,-0.039,35.036,24.533
optimized,optimization,13.435,-0.014,True,0.259,0.403,10.345,0.692,0.247,0.151,0.0,-0.018,37.287,26.114
lhs_top1,dataset,13.665,-0.349,False,0.373,0.833,6.452,0.803,0.149,0.225,25.0,-0.015,43.366,30.370
lhs_top2,dataset,13.507,-0.276,False,0.314,0.722,5.837,0.862,0.205,0.142,25.0,-0.065,41.646,29.181
lhs_top3,dataset,13.354,-0.362,False,0.363,0.727,6.932,0.846,0.157,0.211,25.0,-0.053,41.964,29.378
lhs_top4,dataset,10.212,-0.378,False,0.367,0.875,6.325,0.983,0.151,0.360,25.0,-0.079,47.695,33.386
lhs_top5,dataset,10.088,-0.219,False,0.518,0.517,6.464,0.579,0.240,0.265,25.0,-0.028,25.104,17.592
blend_33,interpolation,12.703,-0.273,False,0.360,0.674,7.509,0.767,0.181,0.201,25.0,-0.015,41.379,28.978
blend_67,interpolation,12.450,-0.157,False,0.322,0.528,8.627,0.729,0.214,0.176,25.0,-0.015,39.312,27.532


## 5. Pareto Plot — L/D vs. Manufacturability

The fundamental trade-off: aerodynamic performance vs. fabrication ease. The Pareto front shows the best achievable compromise at each performance level.

In [6]:
fig = plot_pareto(catalog, save_path='../output/catalog_pareto.png')
plt.show()

<Figure size 768x576 with 0 Axes>

## 6. Radar Chart — Multi-Criteria Comparison

Normalized comparison across 6 dimensions: L/D, stability, manufacturability, volume, endurance, and structural mass.

In [7]:
# Compare key designs: baseline, top LHS designs, blends, and optimized
radar_designs = [n for n in catalog.names if n != 'baseline'][:6]  # skip infeasible baseline
fig = plot_radar(catalog, designs=radar_designs,
                 save_path='../output/catalog_radar.png')
plt.show()

## 7. Geometric Comparison — Planform Overlays

Top view (planform) and front view (dihedral) superimposed for all catalog designs.

In [8]:
fig = plot_planform_overlay(catalog, save_path='../output/catalog_planforms.png')
plt.show()

## 8. Summary Table (visual)

In [9]:
fig = plot_summary_table(catalog, save_path='../output/catalog_table.png')
plt.show()

## 9. Batch STEP Export

Export selected designs as STEP v2 (OML solid) for downstream structural analysis.
Each file is a watertight NURBS solid ready for FEA meshing.

In [10]:
# Select which designs to export (uncomment to run — takes ~30s per design)
# paths = catalog.export_all_step(
#     output_dir='../output/catalog',
#     version='v2',
#     n_profile=100,
# )
# for name, path in paths.items():
#     print(f'  {name}: {path}')

## 10. Save Catalog

Persist the catalog (designs + metrics) as JSON for use by external pipelines (FEA, CFD, manufacturing).

In [11]:
catalog.save('../output/catalog.json')
print(f'Catalog saved: {len(catalog)} designs → output/catalog.json')

# To reload in another session:
# catalog = DesignCatalog.load('../output/catalog.json')

Catalog saved: 9 designs → output/catalog.json
